In [ ]:
import pandas as pd

splits = {'train': 'ViCTSD_train.csv', 'validation': 'ViCTSD_valid.csv', 'test': 'ViCTSD_test.csv'}
df = pd.read_csv("hf://datasets/tarudesu/ViCTSD/" + splits["train"])
# Login using e.g. `huggingface-cli login` to access this dataset
df2 = pd.read_csv("hf://datasets/visolex/ViHSD/ViHSD.csv")
# Login using e.g. `huggingface-cli login` to access this dataset
df3 = pd.read_csv("hf://datasets/visolex/ViSpamReviews/ViSpamReviews.csv")
# Synthetic vietnamese data 
df_ham = pd.read_csv('/kaggle/input/datasets/nguynxunphc/synthetic-vietnamese-data/vietnamese_ham_dataset.csv')
df_spam = pd.read_csv('/kaggle/input/datasets/nguynxunphc/synthetic-vietnamese-data/vietnamese_spam_dataset_kaggle.csv')
# jigsaw toxic comment
train_df = pd.read_csv('/kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge/test.csv')
test_label_df = pd.read_csv('/kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge/test_labels.csv')

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for f in filenames:
        print("   ", f)

In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# LOAD DATASETS
# =========================================================

print("=" * 80)
print("LOADING DATASETS")
print("=" * 80)

# -------------------------
# ViCTSD
# -------------------------
victsd_splits = {
    "train": "ViCTSD_train.csv",
    "validation": "ViCTSD_valid.csv",
    "test": "ViCTSD_test.csv"
}

victsd_df = pd.concat([
    pd.read_csv(f"hf://datasets/tarudesu/ViCTSD/{file}")
    for file in victsd_splits.values()
], ignore_index=True)

print(f"ViCTSD: {victsd_df.shape}")

# -------------------------
# ViHSD
# -------------------------
vihsd_df = pd.read_csv(
    "hf://datasets/visolex/ViHSD/ViHSD.csv"
)

print(f"ViHSD: {vihsd_df.shape}")

# -------------------------
# ViSpamReviews
# -------------------------
vispam_df = pd.read_csv(
    "hf://datasets/visolex/ViSpamReviews/ViSpamReviews.csv"
)

print(f"ViSpamReviews: {vispam_df.shape}")

# -------------------------
# Synthetic Vietnamese
# -------------------------
synthetic_ham_df = pd.read_csv(
    "/kaggle/input/datasets/nguynxunphc/synthetic-vietnamese-data/vietnamese_ham_dataset.csv"
)

synthetic_spam_df = pd.read_csv(
    "/kaggle/input/datasets/nguynxunphc/synthetic-vietnamese-data/vietnamese_spam_dataset_kaggle.csv"
)

print(f"Synthetic Ham : {synthetic_ham_df.shape}")
print(f"Synthetic Spam: {synthetic_spam_df.shape}")

# -------------------------
# Jigsaw Toxic
# -------------------------
jigsaw_train_df = pd.read_csv(
    "/kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge/train.csv"
)

print(f"Jigsaw: {jigsaw_train_df.shape}")


# =========================================================
# STEP 1 + STEP 2
# STANDARDIZE + LABEL MAPPING
# =========================================================

print("\n" + "=" * 80)
print("STANDARDIZING DATASETS")
print("=" * 80)

standardized_dfs = []

# =========================================================
# ViCTSD
# =========================================================

victsd_final = pd.DataFrame({
    "text": victsd_df["Comment"].astype(str),
    "toxic": victsd_df["Toxicity"].fillna(0).astype(int),
    "spam": 0,
    "lang": "vi",
    "source": "ViCTSD"
})

standardized_dfs.append(victsd_final)

print(f"ViCTSD done: {victsd_final.shape}")


# =========================================================
# ViHSD
# label_id:
# 0 = clean
# 1 = offensive
# 2 = hate
# =========================================================

vihsd_final = pd.DataFrame({
    "text": vihsd_df["free_text"].astype(str),
    "toxic": (vihsd_df["label_id"] != 0).astype(int),
    "spam": 0,
    "lang": "vi",
    "source": "ViHSD"
})

standardized_dfs.append(vihsd_final)

print(f"ViHSD done: {vihsd_final.shape}")


# =========================================================
# ViSpamReviews
# SpamLabel:
# 0 = non-spam
# !=0 = spam
# =========================================================

vispam_final = pd.DataFrame({
    "text": vispam_df["Comment"].astype(str),
    "toxic": 0,
    "spam": (vispam_df["SpamLabel"] != 0).astype(int),
    "lang": "vi",
    "source": "ViSpamReviews"
})

standardized_dfs.append(vispam_final)

print(f"ViSpamReviews done: {vispam_final.shape}")


# =========================================================
# Synthetic Ham
# =========================================================

synthetic_ham_final = pd.DataFrame({
    "text": synthetic_ham_df["text"].astype(str),
    "toxic": 0,
    "spam": 0,
    "lang": "vi",
    "source": "SyntheticHam"
})

standardized_dfs.append(synthetic_ham_final)

print(f"SyntheticHam done: {synthetic_ham_final.shape}")


# =========================================================
# Synthetic Spam
# =========================================================

synthetic_spam_final = pd.DataFrame({
    "text": synthetic_spam_df["text"].astype(str),
    "toxic": 0,
    "spam": 1,
    "lang": "vi",
    "source": "SyntheticSpam"
})

standardized_dfs.append(synthetic_spam_final)

print(f"SyntheticSpam done: {synthetic_spam_final.shape}")


# =========================================================
# Jigsaw Toxic
# toxic = any toxic label > 0
# =========================================================

toxic_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

jigsaw_final = pd.DataFrame({
    "text": jigsaw_train_df["comment_text"].astype(str),
    "toxic": (jigsaw_train_df[toxic_cols].sum(axis=1) > 0).astype(int),
    "spam": 0,
    "lang": "en",
    "source": "Jigsaw"
})

standardized_dfs.append(jigsaw_final)

print(f"Jigsaw done: {jigsaw_final.shape}")


# =========================================================
# MERGE ALL
# =========================================================

final_df = pd.concat(
    standardized_dfs,
    ignore_index=True
)

# =========================================================
# BASIC CLEANING
# =========================================================

print("\n" + "=" * 80)
print("BASIC CLEANING")
print("=" * 80)

before = len(final_df)

# remove NaN
final_df = final_df.dropna(subset=["text"])

# strip text
final_df["text"] = final_df["text"].astype(str).str.strip()

# remove empty
final_df = final_df[
    final_df["text"].str.len() > 0
]

# remove exact duplicates
final_df = final_df.drop_duplicates(
    subset=["text", "toxic", "spam"]
)

after = len(final_df)

print(f"Before cleaning : {before}")
print(f"After cleaning  : {after}")
print(f"Removed         : {before - after}")


# =========================================================
# FINAL INFO
# =========================================================

print("\n" + "=" * 80)
print("FINAL DATASET")
print("=" * 80)

print(final_df.head())

print("\nShape:")
print(final_df.shape)

print("\nToxic Distribution:")
print(final_df["toxic"].value_counts())

print("\nSpam Distribution:")
print(final_df["spam"].value_counts())

print("\nLanguage Distribution:")
print(final_df["lang"].value_counts())

print("\nSource Distribution:")
print(final_df["source"].value_counts())


# =========================================================
# OPTIONAL SAVE
# =========================================================

# final_df.to_csv("moderation_dataset.csv", index=False)

print("\nDone.")

In [ ]:
# =========================================================
# STEP 3
# DATA CLEANING + NORMALIZATION
# =========================================================

import re
import unicodedata

print("=" * 80)
print("STEP 3 - CLEANING + NORMALIZATION")
print("=" * 80)

before_cleaning = len(final_df)

# =========================================================
# TEXT CLEANING FUNCTION
# =========================================================

def clean_text(text):
    """
    Minimal cleaning for moderation tasks.
    Keep:
    - emoji
    - slang
    - punctuation
    - casing information (mostly)
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # ---------------------------------
    # unicode normalize
    # ---------------------------------
    text = unicodedata.normalize("NFKC", text)

    # ---------------------------------
    # remove invisible characters
    # ---------------------------------
    text = re.sub(r"[\u200b\u200c\u200d\uFEFF]", "", text)

    # ---------------------------------
    # normalize newlines
    # ---------------------------------
    text = text.replace("\r", "\n")

    # ---------------------------------
    # collapse excessive whitespace
    # ---------------------------------
    text = re.sub(r"\s+", " ", text)

    # ---------------------------------
    # strip
    # ---------------------------------
    text = text.strip()

    return text


# =========================================================
# APPLY CLEANING
# =========================================================

final_df["text"] = final_df["text"].apply(clean_text)

# =========================================================
# REMOVE EMPTY TEXT
# =========================================================

final_df = final_df[
    final_df["text"].str.len() > 0
]

# =========================================================
# TEXT LENGTH FEATURES
# =========================================================

final_df["char_len"] = final_df["text"].str.len()

final_df["word_len"] = final_df["text"].str.split().str.len()

# =========================================================
# REMOVE EXTREME OUTLIERS
# =========================================================

MAX_CHAR_LEN = 2000
MAX_WORD_LEN = 400

final_df = final_df[
    (final_df["char_len"] <= MAX_CHAR_LEN)
]

final_df = final_df[
    (final_df["word_len"] <= MAX_WORD_LEN)
]

# =========================================================
# REMOVE DUPLICATES AGAIN
# after cleaning normalization
# =========================================================

before_dedup = len(final_df)

final_df = final_df.drop_duplicates(
    subset=["text", "toxic", "spam"]
)

after_dedup = len(final_df)

# =========================================================
# OPTIONAL:
# REMOVE VERY SHORT LOW-INFO TEXT
# =========================================================

final_df = final_df[
    final_df["char_len"] >= 2
]

# =========================================================
# RESET INDEX
# =========================================================

final_df = final_df.reset_index(drop=True)

# =========================================================
# FINAL REPORT
# =========================================================

print("\nFINAL SHAPE")
print(final_df.shape)

print("\nREMOVED")
print(f"Initial rows        : {before_cleaning}")
print(f"After dedup         : {after_dedup}")
print(f"Final rows          : {len(final_df)}")

print("\nLABEL DISTRIBUTION")

print("\nTOXIC")
print(final_df["toxic"].value_counts())

print("\nSPAM")
print(final_df["spam"].value_counts())

print("\nLANG")
print(final_df["lang"].value_counts())

print("\nSOURCE")
print(final_df["source"].value_counts())

# =========================================================
# LENGTH STATS
# =========================================================

print("\nTEXT LENGTH")

print("\nCharacters:")
print(final_df["char_len"].describe())

print("\nWords:")
print(final_df["word_len"].describe())

# =========================================================
# SAMPLE CHECKS
# =========================================================

print("\nTOXIC SAMPLES")
print(
    final_df[final_df["toxic"] == 1][["text", "source"]]
    .sample(5, random_state=42)
)

print("\nSPAM SAMPLES")
print(
    final_df[final_df["spam"] == 1][["text", "source"]]
    .sample(5, random_state=42)
)

print("\nNORMAL SAMPLES")
print(
    final_df[
        (final_df["toxic"] == 0) &
        (final_df["spam"] == 0)
    ][["text", "source"]]
    .sample(5, random_state=42)
)

# =========================================================
# OPTIONAL SAVE
# =========================================================

# final_df.to_csv("moderation_dataset_cleaned.csv", index=False)

print("\nSTEP 3 DONE.")

In [ ]:
# =========================================================
# STEP 4 + STEP 5
# REBALANCING DATASET
# =========================================================

print("=" * 80)
print("STEP 4 + 5 - DATASET REBALANCING")
print("=" * 80)

# =========================================================
# TARGETS
# =========================================================

TARGETS = {
    "clean": 60000,
    "toxic": 50000,
    "spam": 40000,
    "toxic_spam": 10000,
}

RANDOM_STATE = 42

# =========================================================
# DEFINE GROUPS
# =========================================================

# clean
clean_df = final_df[
    (final_df["toxic"] == 0) &
    (final_df["spam"] == 0)
]

# toxic only
toxic_df = final_df[
    (final_df["toxic"] == 1) &
    (final_df["spam"] == 0)
]

# spam only
spam_df = final_df[
    (final_df["toxic"] == 0) &
    (final_df["spam"] == 1)
]

# toxic + spam
toxic_spam_df = final_df[
    (final_df["toxic"] == 1) &
    (final_df["spam"] == 1)
]

# =========================================================
# ORIGINAL DISTRIBUTION
# =========================================================

print("\nORIGINAL DISTRIBUTION")
print(f"Clean        : {len(clean_df)}")
print(f"Toxic only   : {len(toxic_df)}")
print(f"Spam only    : {len(spam_df)}")
print(f"Toxic + Spam : {len(toxic_spam_df)}")

# =========================================================
# DOWNSAMPLE CLEAN
# =========================================================

clean_sampled = clean_df.sample(
    n=min(TARGETS["clean"], len(clean_df)),
    random_state=RANDOM_STATE
)

# =========================================================
# UPSAMPLE TOXIC
# =========================================================

toxic_sampled = toxic_df.sample(
    n=TARGETS["toxic"],
    replace=len(toxic_df) < TARGETS["toxic"],
    random_state=RANDOM_STATE
)

# =========================================================
# UPSAMPLE SPAM
# =========================================================

spam_sampled = spam_df.sample(
    n=TARGETS["spam"],
    replace=len(spam_df) < TARGETS["spam"],
    random_state=RANDOM_STATE
)

# =========================================================
# UPSAMPLE TOXIC + SPAM
# =========================================================

if len(toxic_spam_df) > 0:
    toxic_spam_sampled = toxic_spam_df.sample(
        n=TARGETS["toxic_spam"],
        replace=len(toxic_spam_df) < TARGETS["toxic_spam"],
        random_state=RANDOM_STATE
    )
else:
    toxic_spam_sampled = pd.DataFrame(columns=final_df.columns)

# =========================================================
# MERGE BALANCED DATASET
# =========================================================

balanced_df = pd.concat([
    clean_sampled,
    toxic_sampled,
    spam_sampled,
    toxic_spam_sampled
], ignore_index=True)

# =========================================================
# SHUFFLE
# =========================================================

balanced_df = balanced_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

# =========================================================
# FINAL STATS
# =========================================================

print("\n" + "=" * 80)
print("BALANCED DATASET")
print("=" * 80)

print("\nShape:")
print(balanced_df.shape)

# =========================================================
# LABEL DISTRIBUTION
# =========================================================

print("\nTOXIC")
print(balanced_df["toxic"].value_counts())

print("\nSPAM")
print(balanced_df["spam"].value_counts())

print("\nLANG")
print(balanced_df["lang"].value_counts())

print("\nSOURCE")
print(balanced_df["source"].value_counts())

# =========================================================
# COMBINED LABELS
# =========================================================

combined_labels = (
    balanced_df["toxic"].astype(str)
    + "_"
    + balanced_df["spam"].astype(str)
)

print("\nCOMBINED LABELS")
print(combined_labels.value_counts())

# =========================================================
# TEXT LENGTH
# =========================================================

print("\nTEXT LENGTH")

print("\nCharacters:")
print(balanced_df["char_len"].describe())

print("\nWords:")
print(balanced_df["word_len"].describe())

# =========================================================
# SAMPLE CHECKS
# =========================================================

print("\nTOXIC SAMPLES")
print(
    balanced_df[
        (balanced_df["toxic"] == 1)
    ][["text", "source"]]
    .sample(5, random_state=42)
)

print("\nSPAM SAMPLES")
print(
    balanced_df[
        (balanced_df["spam"] == 1)
    ][["text", "source"]]
    .sample(5, random_state=42)
)

print("\nCLEAN SAMPLES")
print(
    balanced_df[
        (balanced_df["toxic"] == 0) &
        (balanced_df["spam"] == 0)
    ][["text", "source"]]
    .sample(5, random_state=42)
)

# =========================================================
# OPTIONAL SAVE
# =========================================================

# balanced_df.to_csv("moderation_dataset_balanced.csv", index=False)

print("\nSTEP 4 + 5 DONE.")

In [ ]:
# =========================================================
# FIXES AFTER STEP 4 + 5
# =========================================================

print("=" * 80)
print("FIXING DATASET ISSUES")
print("=" * 80)

# =========================================================
# FIX 1
# char_len / word_len dtype bug
# =========================================================

balanced_df["char_len"] = balanced_df["char_len"].astype(int)
balanced_df["word_len"] = balanced_df["word_len"].astype(int)

print("\nFixed length column dtypes.")

print("\nCharacter Length Stats:")
print(balanced_df["char_len"].describe())

print("\nWord Length Stats:")
print(balanced_df["word_len"].describe())


# =========================================================
# FIX 2
# Increase Vietnamese clean data
# =========================================================

print("\n" + "=" * 80)
print("ADDING MORE VIETNAMESE CLEAN DATA")
print("=" * 80)

# current clean Vietnamese
vi_clean_df = final_df[
    (final_df["lang"] == "vi") &
    (final_df["toxic"] == 0) &
    (final_df["spam"] == 0)
]

print(f"Available VI clean samples: {len(vi_clean_df)}")

# target additional clean VI samples
TARGET_EXTRA_VI_CLEAN = 15000

extra_vi_clean = vi_clean_df.sample(
    n=min(TARGET_EXTRA_VI_CLEAN, len(vi_clean_df)),
    random_state=42
)

# add
balanced_df = pd.concat(
    [balanced_df, extra_vi_clean],
    ignore_index=True
)

print(f"Added VI clean samples: {len(extra_vi_clean)}")


# =========================================================
# FIX 3
# Create toxic + spam overlap samples
# =========================================================

print("\n" + "=" * 80)
print("CREATING TOXIC + SPAM OVERLAP")
print("=" * 80)

# toxic Vietnamese samples
vi_toxic_df = final_df[
    (final_df["lang"] == "vi") &
    (final_df["toxic"] == 1)
]

# spam Vietnamese samples
vi_spam_df = final_df[
    (final_df["lang"] == "vi") &
    (final_df["spam"] == 1)
]

TARGET_TOXIC_SPAM = 7000

toxic_spam_samples = []

spam_cta_templates = [
    " inbox mình nhé",
    " ib tele @admin nhé",
    " vào nhóm kín đi",
    " tham gia tele kiếm tiền",
    " click link nhận thưởng",
    " nhắn zalo mình nhé",
    " kiếm tiền online ib mình",
]

# generate synthetic toxic+spam
for i in range(TARGET_TOXIC_SPAM):

    toxic_text = vi_toxic_df.sample(1).iloc[0]["text"]

    cta = np.random.choice(spam_cta_templates)

    new_text = toxic_text + " " + cta

    toxic_spam_samples.append({
        "text": new_text,
        "toxic": 1,
        "spam": 1,
        "lang": "vi",
        "source": "SyntheticToxicSpam",
        "char_len": len(new_text),
        "word_len": len(new_text.split())
    })

toxic_spam_df = pd.DataFrame(toxic_spam_samples)

print(f"Created toxic+spam samples: {len(toxic_spam_df)}")

# add
balanced_df = pd.concat(
    [balanced_df, toxic_spam_df],
    ignore_index=True
)


# =========================================================
# FINAL SHUFFLE
# =========================================================

balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# =========================================================
# FINAL REPORT
# =========================================================

print("\n" + "=" * 80)
print("FINAL DATASET AFTER FIXES")
print("=" * 80)

print("\nShape:")
print(balanced_df.shape)

# =========================================================
# LABELS
# =========================================================

print("\nTOXIC")
print(balanced_df["toxic"].value_counts())

print("\nSPAM")
print(balanced_df["spam"].value_counts())

# =========================================================
# COMBINED LABELS
# =========================================================

combined_labels = (
    balanced_df["toxic"].astype(str)
    + "_"
    + balanced_df["spam"].astype(str)
)

print("\nCOMBINED LABELS")
print(combined_labels.value_counts())

# =========================================================
# LANG
# =========================================================

print("\nLANG")
print(balanced_df["lang"].value_counts())

# =========================================================
# SOURCE
# =========================================================

print("\nSOURCE")
print(balanced_df["source"].value_counts())

# =========================================================
# LENGTH STATS
# =========================================================

print("\nTEXT LENGTH")

print("\nCharacters:")
print(balanced_df["char_len"].describe())

print("\nWords:")
print(balanced_df["word_len"].describe())

# =========================================================
# SAMPLE CHECKS
# =========================================================

print("\nTOXIC + SPAM SAMPLES")

print(
    balanced_df[
        (balanced_df["toxic"] == 1) &
        (balanced_df["spam"] == 1)
    ][["text"]]
    .sample(5, random_state=42)
)

# =========================================================
# OPTIONAL SAVE
# =========================================================

# balanced_df.to_csv(
#     "moderation_dataset_final.csv",
#     index=False
# )

print("\nFIXES COMPLETED.")

In [ ]:
# =========================================================
# SAVE FINAL DATASET
# =========================================================

OUTPUT_PATH = "/kaggle/working/moderation_dataset_v1.csv"

balanced_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("=" * 80)
print("DATASET SAVED")
print("=" * 80)

print(f"Saved to: {OUTPUT_PATH}")
print(f"Shape: {balanced_df.shape}")

# =========================================================
# QUICK VERIFY
# =========================================================

loaded_df = pd.read_csv(OUTPUT_PATH)

print("\nVERIFY LOADED DATASET")
print(loaded_df.head())

print("\nLoaded Shape:")
print(loaded_df.shape)

# =========================================================
# KAGGLE DOWNLOAD NOTE
# =========================================================

print("\nFile is ready for download from Kaggle output panel.")

In [ ]:
# =========================================================
# STEP 6
# TRAIN / VALID / TEST SPLIT
# =========================================================

from sklearn.model_selection import train_test_split

print("=" * 80)
print("STEP 6 - DATASET SPLIT")
print("=" * 80)

# =========================================================
# CREATE STRATIFY LABEL
# =========================================================

balanced_df["label_combo"] = (
    balanced_df["toxic"].astype(str)
    + "_"
    + balanced_df["spam"].astype(str)
)

print("\nCombined Labels:")
print(balanced_df["label_combo"].value_counts())

# =========================================================
# TRAIN = 80%
# TEMP  = 20%
# =========================================================

train_df, temp_df = train_test_split(
    balanced_df,
    test_size=0.2,
    stratify=balanced_df["label_combo"],
    random_state=42
)

# =========================================================
# VALID = 10%
# TEST  = 10%
# =========================================================

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label_combo"],
    random_state=42
)

# =========================================================
# REMOVE HELPER COLUMN
# =========================================================

for df in [train_df, valid_df, test_df]:
    df.drop(columns=["label_combo"], inplace=True)

# =========================================================
# RESET INDEX
# =========================================================

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# =========================================================
# REPORT
# =========================================================

print("\nDATASET SHAPES")
print(f"Train: {train_df.shape}")
print(f"Valid: {valid_df.shape}")
print(f"Test : {test_df.shape}")

# =========================================================
# LABEL DISTRIBUTION CHECK
# =========================================================

def show_distribution(df, name):

    print("\n" + "-" * 50)
    print(name)
    print("-" * 50)

    print("\nTOXIC")
    print(df["toxic"].value_counts(normalize=True))

    print("\nSPAM")
    print(df["spam"].value_counts(normalize=True))

show_distribution(train_df, "TRAIN")
show_distribution(valid_df, "VALID")
show_distribution(test_df, "TEST")

print("\nSTEP 6 DONE.")

In [ ]:
# =========================================================
# STEP 7
# TOKENIZATION + DATASET PREPARATION
# =========================================================

!pip install -q transformers datasets accelerate

import torch
from datasets import Dataset
from transformers import AutoTokenizer

print("=" * 80)
print("STEP 7 - TOKENIZATION")
print("=" * 80)

# =========================================================
# MODEL CHECKPOINT
# =========================================================

MODEL_NAME = "xlm-roberta-base"

# Alternative:
# MODEL_NAME = "vinai/phobert-base"

print(f"\nUsing model: {MODEL_NAME}")

# =========================================================
# TOKENIZER
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# =========================================================
# CONVERT TO HF DATASET
# =========================================================

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

print("\nHF Dataset created.")

# =========================================================
# TOKENIZATION FUNCTION
# =========================================================

MAX_LENGTH = 256

def tokenize_function(example):

    encoding = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    # multi-label targets
    encoding["labels"] = [
        float(example["toxic"]),
        float(example["spam"])
    ]

    return encoding

# =========================================================
# TOKENIZE
# =========================================================

train_dataset = train_dataset.map(
    tokenize_function
)

valid_dataset = valid_dataset.map(
    tokenize_function
)

test_dataset = test_dataset.map(
    tokenize_function
)

# =========================================================
# REMOVE UNUSED COLUMNS
# =========================================================

KEEP_COLUMNS = [
    "input_ids",
    "attention_mask",
    "labels"
]

# phobert may need token_type_ids
if "token_type_ids" in train_dataset.column_names:
    KEEP_COLUMNS.append("token_type_ids")

train_dataset = train_dataset.remove_columns([
    col for col in train_dataset.column_names
    if col not in KEEP_COLUMNS
])

valid_dataset = valid_dataset.remove_columns([
    col for col in valid_dataset.column_names
    if col not in KEEP_COLUMNS
])

test_dataset = test_dataset.remove_columns([
    col for col in test_dataset.column_names
    if col not in KEEP_COLUMNS
])

# =========================================================
# TORCH FORMAT
# =========================================================

train_dataset.set_format("torch")
valid_dataset.set_format("torch")
test_dataset.set_format("torch")

# =========================================================
# CHECK SAMPLE
# =========================================================

print("\nTOKENIZED SAMPLE")
print(train_dataset[0])

print("\nDataset Ready.")

print(f"\nTrain size: {len(train_dataset)}")
print(f"Valid size: {len(valid_dataset)}")
print(f"Test size : {len(test_dataset)}")

print("\nSTEP 7 DONE.")

In [ ]:
# =========================================================
# STEP 8 (FIXED)
# MODEL TRAINING
# =========================================================

!pip install -q transformers accelerate evaluate scikit-learn

import numpy as np
import torch
import evaluate

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print("=" * 80)
print("STEP 8 - MODEL TRAINING (FIXED)")
print("=" * 80)

# =========================================================
# DEVICE
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nDevice: {device}")

# =========================================================
# MODEL
# =========================================================

NUM_LABELS = 2

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

model.to(device)

print("\nModel loaded.")

# =========================================================
# METRICS
# =========================================================

f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

# =========================================================
# THRESHOLD
# =========================================================

THRESHOLD = 0.5

# =========================================================
# COMPUTE METRICS
# =========================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = torch.sigmoid(
        torch.tensor(logits)
    ).numpy()

    preds = (probs >= THRESHOLD).astype(int)

    results = {}

    # -----------------------------------------------------
    # TOXIC
    # -----------------------------------------------------

    toxic_f1 = f1_metric.compute(
        predictions=preds[:, 0],
        references=labels[:, 0],
        average="binary"
    )

    toxic_precision = precision_metric.compute(
        predictions=preds[:, 0],
        references=labels[:, 0],
        average="binary"
    )

    toxic_recall = recall_metric.compute(
        predictions=preds[:, 0],
        references=labels[:, 0],
        average="binary"
    )

    # -----------------------------------------------------
    # SPAM
    # -----------------------------------------------------

    spam_f1 = f1_metric.compute(
        predictions=preds[:, 1],
        references=labels[:, 1],
        average="binary"
    )

    spam_precision = precision_metric.compute(
        predictions=preds[:, 1],
        references=labels[:, 1],
        average="binary"
    )

    spam_recall = recall_metric.compute(
        predictions=preds[:, 1],
        references=labels[:, 1],
        average="binary"
    )

    # -----------------------------------------------------
    # STORE RESULTS
    # -----------------------------------------------------

    results["toxic_f1"] = toxic_f1["f1"]
    results["toxic_precision"] = toxic_precision["precision"]
    results["toxic_recall"] = toxic_recall["recall"]

    results["spam_f1"] = spam_f1["f1"]
    results["spam_precision"] = spam_precision["precision"]
    results["spam_recall"] = spam_recall["recall"]

    results["macro_f1"] = (
        results["toxic_f1"]
        + results["spam_f1"]
    ) / 2

    return results

# =========================================================
# TRAINING ARGUMENTS
# =========================================================

training_args = TrainingArguments(
    output_dir="./moderation_model",

    # training
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,
    weight_decay=0.01,

    # evaluation
    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=valid_dataset,

    compute_metrics=compute_metrics
)

# =========================================================
# TRAIN
# =========================================================

trainer.train()

print("\nSTEP 8 DONE.")

In [ ]:
# =========================================================
# STEP 9 (FIXED)
# FINAL EVALUATION
# =========================================================

import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print("=" * 80)
print("STEP 9 - FINAL EVALUATION (FIXED)")
print("=" * 80)

# =========================================================
# PREDICT
# =========================================================

predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels = predictions.label_ids

# =========================================================
# PROBABILITIES
# =========================================================

probs = torch.sigmoid(
    torch.tensor(logits)
).numpy()

preds = (probs >= THRESHOLD).astype(int)

# =========================================================
# TOXIC REPORT
# =========================================================

print("\n" + "=" * 80)
print("TOXIC CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        labels[:, 0],
        preds[:, 0],
        digits=4
    )
)

# =========================================================
# SPAM REPORT
# =========================================================

print("\n" + "=" * 80)
print("SPAM CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        labels[:, 1],
        preds[:, 1],
        digits=4
    )
)

# =========================================================
# CONFUSION MATRICES
# =========================================================

print("\n" + "=" * 80)
print("TOXIC CONFUSION MATRIX")
print("=" * 80)

print(
    confusion_matrix(
        labels[:, 0],
        preds[:, 0]
    )
)

print("\n" + "=" * 80)
print("SPAM CONFUSION MATRIX")
print("=" * 80)

print(
    confusion_matrix(
        labels[:, 1],
        preds[:, 1]
    )
)

# =========================================================
# SAMPLE PREDICTIONS
# =========================================================

print("\n" + "=" * 80)
print("SAMPLE PREDICTIONS")
print("=" * 80)

sample_df = test_df.copy()

sample_df["toxic_pred"] = preds[:, 0]
sample_df["spam_pred"] = preds[:, 1]

sample_df["toxic_prob"] = probs[:, 0]
sample_df["spam_prob"] = probs[:, 1]

samples = sample_df.sample(
    10,
    random_state=42
)

for idx, row in samples.iterrows():

    print("\n" + "-" * 60)

    print(f"TEXT:\n{row['text']}")

    print("\nTRUE LABELS:")
    print(
        f"Toxic={row['toxic']} | "
        f"Spam={row['spam']}"
    )

    print("\nPREDICTIONS:")
    print(
        f"Toxic={row['toxic_pred']} "
        f"({row['toxic_prob']:.4f})"
    )

    print(
        f"Spam={row['spam_pred']} "
        f"({row['spam_prob']:.4f})"
    )

# =========================================================
# SAVE MODEL
# =========================================================

SAVE_PATH = "./final_moderation_model"

trainer.save_model(SAVE_PATH)

# tokenizer save
tokenizer.save_pretrained(SAVE_PATH)

print("\n" + "=" * 80)
print("MODEL SAVED")
print("=" * 80)

print(f"\nSaved to: {SAVE_PATH}")

print("\nSTEP 9 DONE.")